# SANPO-Real Dataset Preparation for Google Colab
This notebook downloads a subset of the SANPO-Real dataset from Google Cloud Storage (GCS) and saves it directly to your mounted Google Drive.

By using Colab's native Drive mount, it writes files directly to your Drive via standard filesystem operations. This bypasses the Google Drive API completely, avoiding network rate limits, multithreading SSL socket errors, and local disk space limitations.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
from __future__ import annotations

import gzip
import io
import json
import random
import concurrent.futures
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional
import os
import time
import threading

import numpy as np
from google.cloud import storage

# ── Configuration ─────────────────────────────────────────────────────────
# Path to the target directory inside your Google Drive
# Make sure this directory exists or it will be created automatically.
GDRIVE_OUTPUT_DIR = "/content/drive/MyDrive/sanpo_real"

MAX_SESSIONS = 500       # Total sessions to sample
BATCH_SIZE = 50          # Batch size to download & save in each iteration
SIZE_LIMIT_GB = 100.0    # Safety size limit in GB

In [ ]:
@dataclass
class SamplerConfig:
    bucket_name: str = "gresearch"
    prefix: str = "sanpo_dataset/v0/sanpo-real/"
    max_sessions: int = 500
    batch_size: int = 50
    max_sample: int = 100        # frame triples per session per camera
    stride: int = 15             # take 1 frame every N frames
    seed: int = 42
    cameras: List[str] = field(default_factory=lambda: ["head", "chest"])
    require_all_modalities: bool = True
    size_limit_gb: float = 100.0


class SanpoDepthPairSampler:
    def __init__(self, config: SamplerConfig):
        self.cfg = config
        self.client = storage.Client.create_anonymous_client()
        self.rng = random.Random(config.seed)

    def _gs_uri(self, object_name: str) -> str:
        return f"gs://{self.cfg.bucket_name}/{object_name}"

    def _session_prefix(self, session_id: str) -> str:
        return f"{self.cfg.prefix.rstrip('/')}/{session_id}/"

    def _calib_url(self, session_id: str) -> str:
        return self._gs_uri(f"{self._session_prefix(session_id)}description.json")

    def _list_prefixes(self, prefix: str) -> List[str]:
        it = self.client.list_blobs(
            self.cfg.bucket_name, prefix=prefix, delimiter="/"
        )
        prefixes = set()
        for page in it.pages:
            prefixes.update(page.prefixes)
        return sorted(prefixes)

    def _list_files_flat(self, prefix: str) -> List[str]:
        it = self.client.list_blobs(
            self.cfg.bucket_name, prefix=prefix, delimiter="/"
        )
        out = []
        for page in it.pages:
            for blob in page:
                base = blob.name.rsplit("/", 1)[-1]
                if base and not base.endswith("$folder$"):
                    out.append(blob.name)
        return sorted(out)

    def _normalize_id(self, path: str) -> str:
        name = path.rsplit("/", 1)[-1]
        if name.endswith(".float16.gz"): return name[:-11]
        return name.rsplit(".", 1)[0] if "." in name else name

    def _frame_map(self, session_id: str, rel_prefix: str) -> Dict[str, str]:
        prefix = f"{self._session_prefix(session_id)}{rel_prefix.strip('/')}/"
        files = self._list_files_flat(prefix)
        return {self._normalize_id(f): f for f in files}

    def _sample_camera(self, left_map, right_map, depth_map) -> List[dict]:
        sets = [set(m.keys()) for m in (left_map, right_map, depth_map) if m]
        if not sets: return []
        valid_ids = sorted(
            set.intersection(*sets) if self.cfg.require_all_modalities
            else set.union(*sets)
        )
        if not valid_ids: return []
        strided = valid_ids[:: max(1, self.cfg.stride)]
        chosen = self.rng.sample(strided, k=min(self.cfg.max_sample, len(strided)))
        return [
            {
                "id": fid,
                "left":  self._gs_uri(left_map[fid])  if fid in left_map  else None,
                "right": self._gs_uri(right_map[fid]) if fid in right_map else None,
                "depth": self._gs_uri(depth_map[fid]) if fid in depth_map else None,
            }
            for fid in chosen
        ]

    def dry_run(self, ignored_sessions: Optional[List[str]] = None, num_sessions_to_select: Optional[int] = None) -> List[dict]:
        all_prefixes = self._list_prefixes(self.cfg.prefix.rstrip("/") + "/")
        session_ids = [p.rstrip("/").split("/")[-1] for p in all_prefixes]
        if not session_ids: return []

        ignored_set = set(ignored_sessions or [])
        available = [s for s in session_ids if s not in ignored_set]
        if not available:
            print("[sampler] All sessions have been processed.")
            return []

        k = num_sessions_to_select or self.cfg.batch_size
        chosen = self.rng.sample(available, k=min(k, len(available)))
        CAM_DIRS = {"head": "camera_head", "chest": "camera_chest"}
        results = []

        for sid in chosen:
            entry = {
                "session_name": sid,
                "calib_url": self._calib_url(sid),
            }
            for cam in self.cfg.cameras:
                cdir = CAM_DIRS[cam]
                entry[cam] = self._sample_camera(
                    self._frame_map(sid, f"{cdir}/left/video_frames"),
                    self._frame_map(sid, f"{cdir}/right/video_frames"),
                    self._frame_map(sid, f"{cdir}/left/depth_maps"),
                )
            results.append(entry)

        total = sum(len(r.get(c, [])) for r in results for c in self.cfg.cameras)
        print(f"[sampler] Chosen {len(results)} sessions | {total} frame triples")
        return results

In [ ]:
def _extract_calib(desc: dict, camera: str) -> dict:
    cam_key = "camera_head" if camera == "head" else "camera_chest"
    locations = desc.get("session_camera_location", [])
    details   = desc.get("session_camera_details", [])
    cam_detail = next(
        (details[i] for i, loc in enumerate(locations) if loc == cam_key and i < len(details)),
        details[1 if camera == "head" else 0] if details else None,
    )
    if not cam_detail: return {}
    lp = cam_detail["left_camera_params"]
    return {
        "camera":           cam_key,
        "focal_length_px":  lp["fx"],
        "baseline_m":       round(abs(cam_detail["stereo_transform"]["coeff"][3]) / 1000.0, 8),
        "cx":               lp["cx"],
        "cy":               lp["cy"],
        "image_width":      lp["image_width"],
        "image_height":     lp["image_height"],
        "fps":              cam_detail.get("fps"),
        "model":            cam_detail.get("model"),
    }


def _decode_float16_gz(raw: bytes) -> np.ndarray:
    with gzip.open(io.BytesIO(raw), "rb") as f:
        data = np.frombuffer(f.read(), dtype=np.float16)
    h = int(data[0])
    w = int(data[1])
    return data[2:].reshape(h, w).astype(np.float32)


def _load_session_index(output_dir: str) -> Dict[str, str]:
    index_file = Path(output_dir) / "session_index.json"
    if index_file.exists():
        with open(index_file, "r") as f:
            return json.load(f)
    return {}


def _save_session_index(output_dir: str, index: Dict[str, str]) -> None:
    index_file = Path(output_dir) / "session_index.json"
    index_file.parent.mkdir(parents=True, exist_ok=True)
    with open(index_file, "w") as f:
        json.dump(index, f, indent=2)


def _next_session_number(index: Dict[str, str]) -> int:
    if not index:
        return 1
    existing = [int(v.split("_")[1]) for v in index.values() if "_" in v]
    return max(existing) + 1 if existing else 1


def _load_processed_sessions(output_dir: str) -> List[str]:
    p_file = Path(output_dir) / "processed_sessions.json"
    if p_file.exists():
        with open(p_file, "r") as f:
            return json.load(f)
    return []


def _save_processed_sessions(output_dir: str, processed: List[str]) -> None:
    p_file = Path(output_dir) / "processed_sessions.json"
    p_file.parent.mkdir(parents=True, exist_ok=True)
    with open(p_file, "w") as f:
        json.dump(processed, f, indent=2)


class SafeCounter:
    def __init__(self):
        self.value = 0
        self.lock = threading.Lock()
    def add(self, n):
        with self.lock:
            self.value += n
    def get(self):
        with self.lock:
            return self.value

In [ ]:
def run_local_download_save(
    results: List[dict],
    output_dir: str,
    bucket_name: str = "gresearch",
    max_workers: int = 8,
    cameras: Optional[List[str]] = None,
    counter: Optional[SafeCounter] = None,
) -> None:
    """
    Tải từ GCS và ghi trực tiếp vào thư mục Google Drive đã mount ở output_dir
    bằng cách sử dụng các hàm API hệ thống tệp cục bộ chuẩn (Local File System API),
    hoàn toàn bỏ qua API HTTP của Google Drive để tránh lỗi quá tải mạng và SSL.
    """
    if cameras is None:
        cameras = [c for c in ["head", "chest"] if any(c in r for r in results)]

    client = storage.Client.create_anonymous_client()

    def fetch(uri: str) -> bytes:
        obj = uri[len(f"gs://{bucket_name}/"):]
        return client.bucket(bucket_name).blob(obj).download_as_bytes()

    # ── Assign session folder mapping ──────────────────────────────────────────
    index = _load_session_index(output_dir)
    next_num = _next_session_number(index)

    for s in results:
        sid = s["session_name"]
        for cam in cameras:
            if not s.get(cam):
                continue
            key = f"{sid}_{cam}"
            if key not in index:
                index[key] = f"session_{next_num:04d}"
                next_num += 1
    _save_session_index(output_dir, index)

    # ── Pre-create Google Drive folder structure (standard directory calls) ────
    print("[filesystem] Pre-creating folder layout on Google Drive...")
    for s in results:
        sid = s["session_name"]
        desc = None
        if s.get("calib_url"):
            try:
                desc = json.loads(fetch(s["calib_url"]))
            except Exception as e:
                print(f"[WARN] calib fetch failed {sid}: {e}")

        for cam in cameras:
            if not s.get(cam):
                continue
            key = f"{sid}_{cam}"
            folder_name = index[key]
            
            # Setup folder paths inside the mounted Drive directory
            sess_dir = Path(output_dir) / folder_name
            left_dir = sess_dir / "left"
            right_dir = sess_dir / "right"
            depth_dir = sess_dir / "depth_ml"
            
            # Create directories physically
            left_dir.mkdir(parents=True, exist_ok=True)
            right_dir.mkdir(parents=True, exist_ok=True)
            depth_dir.mkdir(parents=True, exist_ok=True)
            
            # Write calib.json directly to Drive
            if desc:
                try:
                    calib = _extract_calib(desc, cam)
                    calib_path = sess_dir / "calib.json"
                    with open(calib_path, "w") as f:
                        json.dump(calib, f, indent=2)
                except Exception as e:
                    print(f"[WARN] Failed to write calib.json for {folder_name}: {e}")

    # ── Build task list ────────────────────────────────────────────────────────
    tasks = []
    for s in results:
        sid = s["session_name"]
        for cam in cameras:
            if not s.get(cam):
                continue
            key = f"{sid}_{cam}"
            folder_name = index[key]
            sess_dir = Path(output_dir) / folder_name
            for frame in s.get(cam, []):
                tasks.append((frame, sess_dir))

    total = len(tasks)
    print(f"[pipeline] Starting download and direct write for {total} frames using {max_workers} threads...")

    # ── Parallel fetch & local filesystem write ────────────────────────────────
    done = errors = 0

    def process(task):
        nonlocal done, errors
        frame, sess_dir = task
        fid = frame["id"]
        written_bytes = 0
        try:
            # 1. Left image
            if frame.get("left"):
                left_data = fetch(frame["left"])
                left_path = sess_dir / "left" / f"{fid}.png"
                with open(left_path, "wb") as f:
                    f.write(left_data)
                written_bytes += len(left_data)

            # 2. Right image
            if frame.get("right"):
                right_data = fetch(frame["right"])
                right_path = sess_dir / "right" / f"{fid}.png"
                with open(right_path, "wb") as f:
                    f.write(right_data)
                written_bytes += len(right_data)

            # 3. Depth map
            if frame.get("depth"):
                depth_data = fetch(frame["depth"])
                arr = _decode_float16_gz(depth_data)
                depth_path = sess_dir / "depth_ml" / f"{fid}.npy"
                np.save(depth_path, arr)
                written_bytes += len(depth_data)

            if counter:
                counter.add(written_bytes)
        except Exception as e:
            print(f"[ERROR] Pipeline filesystem write failed for frame {fid}: {e}")
            raise e

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(process, t): t for t in tasks}
        for fut in concurrent.futures.as_completed(futures):
            done += 1
            if fut.exception():
                errors += 1
            if done % 50 == 0 or done == total:
                print(f"  [progress] {done}/{total} files saved  errors={errors}")

    print(f"[done] Completed batch filesystem copy successfully.")


In [ ]:
import math
from IPython.display import clear_output

cfg = SamplerConfig(
    max_sessions=MAX_SESSIONS,
    batch_size=BATCH_SIZE,
    max_sample=100,
    stride=15,
    cameras=["head", "chest"],
    size_limit_gb=SIZE_LIMIT_GB
)

sampler = SanpoDepthPairSampler(cfg)

# Load ignored/processed session list
processed_sessions = _load_processed_sessions(GDRIVE_OUTPUT_DIR)
print(f"Already processed: {len(processed_sessions)} sessions.")

total_processed_bytes = 0
batch_idx = len(processed_sessions) // BATCH_SIZE + 1
batch_statuses = [f"Batch {i+1:02d}: COMPLETED (Restored from index)" for i in range(len(processed_sessions) // BATCH_SIZE)]

while len(processed_sessions) < MAX_SESSIONS:
    total_processed_gb = total_processed_bytes / (1024**3)
    if total_processed_gb >= SIZE_LIMIT_GB:
        print(f"Safety limit reached: {total_processed_gb:.2f} GB / {SIZE_LIMIT_GB:.2f} GB. Stopping.")
        break
        
    sessions_remaining = MAX_SESSIONS - len(processed_sessions)
    if sessions_remaining <= 0:
        break
        
    num_sessions_to_select = min(BATCH_SIZE, sessions_remaining)
    
    # Render Dashboard
    clear_output(wait=True)
    print("============================================================")
    print("         SANPO REAL COLAB DRIVE BATCH PROCESSOR")
    print("============================================================")
    print(f"Configured Max Sessions : {MAX_SESSIONS} (Batch Size: {BATCH_SIZE})")
    print(f"Size Limit              : {SIZE_LIMIT_GB:.2f} GB")
    print(f"Processed Sessions      : {len(processed_sessions)} / {MAX_SESSIONS}")
    print(f"Total Transferred Size  : {total_processed_gb:.4f} GB")
    print("------------------------------------------------------------")
    print("Batch History:")
    for status in batch_statuses:
        print(f"  - {status}")
    print("------------------------------------------------------------")
    print(f"Current Batch {batch_idx:02d} (Selecting {num_sessions_to_select} sessions)...")
    print("============================================================\n")
    
    # 1. Sample sessions for this batch
    results = sampler.dry_run(ignored_sessions=processed_sessions, num_sessions_to_select=num_sessions_to_select)
    if not results:
        print("No more sessions available to process.")
        break
        
    # 2. Run local download and filesystem copy to Drive
    print(f"\n[Batch {batch_idx:02d}] Executing download-to-drive pipeline...")
    counter = SafeCounter()
    try:
        run_local_download_save(
            results=results,
            output_dir=GDRIVE_OUTPUT_DIR,
            bucket_name=cfg.bucket_name,
            max_workers=8,
            counter=counter
        )
        upload_success = True
        batch_bytes = counter.get()
        batch_gb = batch_bytes / (1024**3)
        total_processed_bytes += batch_bytes
    except Exception as e:
        print(f"[ERROR] Pipeline failed for batch {batch_idx:02d}: {e}")
        upload_success = False
        batch_gb = 0.0
        
    # 3. Record progress
    if upload_success:
        for s in results:
            processed_sessions.append(s["session_name"])
        _save_processed_sessions(GDRIVE_OUTPUT_DIR, processed_sessions)
        status_msg = f"Batch {batch_idx:02d}: COMPLETED (Transferred {batch_gb:.4f} GB)"
    else:
        status_msg = f"Batch {batch_idx:02d}: FAILED"
        
    batch_statuses.append(status_msg)
    batch_idx += 1

# Final status print
clear_output(wait=True)
print("============================================================")
print("         SANPO REAL BATCH PROCESSOR COMPLETED")
print("============================================================")
print(f"Processed Sessions      : {len(processed_sessions)} / {MAX_SESSIONS}")
print(f"Total Transferred Size  : {total_processed_bytes / (1024**3):.4f} GB")
print("------------------------------------------------------------")
print("Final Batch Statuses:")
for status in batch_statuses:
    print(f"  - {status}")
print("============================================================")
